# Task 2: Baseline Modeling-Traditional Machine Learning

**Project:** Domain-Adaptive Sentiment & Public Opinion Analysis using Transformers and LLMs  
**Task:** Establish baseline performance using classical ML models (Logistic Regression, SVM, Naïve Bayes) with TF-IDF features.  
**Semester:** Semester 1 2026

---

## Overview

This notebook implements **Task 2: Baseline Modeling** for a domain-adaptive sentiment analysis project. The goal is to quantify how much training data composition alone — with no change to model architecture — affects classification performance on a specialised financial text dataset.

We train three classical machine learning classifiers under three distinct training data conditions (Models A, B, and C), all evaluated on the same held-out Financial PhraseBank test set. This controlled experimental design isolates the effect of domain adaptation from confounding factors such as model complexity or hyperparameter choices.

---

## Experimental Framework (A / B / C)

| Model | Training Data | Testing Data | Purpose |
|-------|--------------|--------------|------|
| **Model A** | GoEmotions (General only) | FPB Test | Measure zero-shot domain transfer |
| **Model B** | FPB Train (Domain only) | FPB Test | Measure performance on limited domain data |
| **Model C** | GoEmotions + FPB Train (Mixed) | FPB Test | Evaluate combined dataset benefit |

**Architecture is identical across A/B/C — only the training data changes.**

---

## Classifiers

| Classifier | Type | Handles imbalance? |
|------------|------|--------------------|
| Logistic Regression | Linear probabilistic | Yes (`class_weight=balanced`) |
| Linear SVM | Linear discriminative | Yes (`class_weight=balanced`) |
| Naïve Bayes (ComplementNB) | Probabilistic generative | Partially (Complement NB design) |

---

## Evaluation Metrics

All models are evaluated using:
- **Accuracy** — proportion of correct predictions
- **F1 (Macro)** — unweighted average F1 across all five classes; reveals minority-class performance
- **F1 (Weighted)** — support-weighted F1; reflects real-world utility on imbalanced data
- **Precision & Recall (Macro)** — complementary diagnostics
- **Per-class F1** — essential for understanding Fear and Joy failure modes
- **Confusion matrices** — visualise misclassification patterns

---

## Research Question

> **How does training data choice influence model performance in domain-adaptive sentiment analysis?**

This question is answered experimentally by holding architecture constant and varying only training data.

## 1. Setup & Imports

All required libraries are imported here. The notebook uses:

- **pandas / numpy** — data loading and numerical operations
- **matplotlib / seaborn** — all visualisations
- **scikit-learn** — TF-IDF vectorisation, classifiers, and evaluation metrics
- **re / string** — text normalisation utilities

`warnings.filterwarnings('ignore')` suppresses convergence warnings from LinearSVC that are expected with short `max_iter` settings on large datasets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)

import re, string
from collections import Counter

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print('All imports successful!')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

All imports successful!
pandas 2.3.3 | numpy 2.3.5


## 2. Load Datasets

### Dataset Descriptions

#### GoEmotions (General Domain)
- **Source:** Google Research — 58,009 Reddit comments annotated with 27 fine-grained emotions, remapped here to 5 classes
- **Role:** Source dataset for general language understanding (Model A training, Model C component)
- **Text style:** Informal, conversational, social-media language
- **Size used:** 43,404 samples after 5-class remapping

#### Financial PhraseBank (FPB)
- **Source:** Malo et al. (2014) — financial news sentences annotated by domain experts
- **Role:** Target domain dataset — training (B, C), validation, and evaluation
- **Text style:** Formal financial news; technical vocabulary
- **Pre-split sizes:** Train 3,392 | Val 727 | Test 727

The **FPB test set is never used for training or validation** — it is the fixed evaluation target for all nine experiments.

Set `BASE` to the folder containing all CSV files if running locally.

In [ ]:
# Set BASE to the folder containing the CSV files.
# If your CSVs (fpb_train.csv, fpb_test.csv, fpb_val.csv, goemotions_5class.csv)
# are in the repo root and this notebook is also in the root, use './'.
# If the notebook is in a subfolder (e.g. baseline/), use '../'.
BASE = './'  # change to '../' if notebook is in a subfolder

goemotions = pd.read_csv(BASE + '../data/processed/goemotions_5class.csv')   # General domain
fpb_train  = pd.read_csv(BASE + '../data/processed/fpb_train.csv')           # FPB train split
fpb_val    = pd.read_csv(BASE + '../data/processed/fpb_val.csv')             # FPB validation split
fpb_test   = pd.read_csv(BASE + '../data/processed/fpb_test.csv')            # FPB test — evaluation target only

print('Dataset Sizes')
print('=' * 50)
print(f'GoEmotions (general):  {len(goemotions):>6,} rows')
print(f'FPB train (domain):    {len(fpb_train):>6,} rows')
print(f'FPB val:               {len(fpb_val):>6,} rows')
print(f'FPB test:              {len(fpb_test):>6,} rows')
print()
print('Columns:', goemotions.columns.tolist())
display(goemotions.head(3))
display(fpb_train.head(3))

### 2.1 Dataset Statistics — At a Glance

The table below summarises the key properties of each dataset before any processing. Understanding these figures is essential for interpreting model performance — a model trained on 43,404 samples that is beaten by one trained on 3,392 is telling us something fundamental about domain mismatch.


In [ ]:
summary_stats = []
for name, df, role in [
    ('GoEmotions', goemotions, 'Model A train / Model C component'),
    ('FPB Train',  fpb_train,  'Model B & C train'),
    ('FPB Val',    fpb_val,    'Validation reference'),
    ('FPB Test',   fpb_test,   'Fixed evaluation target (all models)'),
]:
    lengths = df['text'].dropna().str.split().apply(len)
    summary_stats.append({
        'Dataset':        name,
        'Role':           role,
        'Rows':           f'{len(df):,}',
        'Avg words':      f'{lengths.mean():.1f}',
        'Med words':      f'{lengths.median():.0f}',
        'Max words':      f'{lengths.max()}',
        'Unique labels':  df['label'].nunique(),
        'Missing text':   df['text'].isnull().sum(),
    })

stats_df = pd.DataFrame(summary_stats)
display(stats_df.style.set_caption('Dataset Overview').hide(axis='index'))
print(f'\nTrain/Test ratio for FPB: {len(fpb_train)/len(fpb_test):.1f}x')
print(f'GoEmotions:FPB train ratio: {len(goemotions)/len(fpb_train):.1f}x (key imbalance in Model C)')

## 3. Exploratory Data Analysis (EDA)

Before modelling, we examine the datasets in depth to understand their characteristics. This section covers:

1. Label distributions and class imbalance
2. Text length distributions
3. Vocabulary overlap between domains
4. Sample text inspection by class

Understanding these properties motivates key modelling decisions: why `class_weight='balanced'` is necessary, why bigrams matter, and why Model A is expected to underperform on FPB.

### 3.1 Label Distributions

Label imbalance is one of the most important dataset properties to understand before training. A naive model that always predicts **Neutral** would reach approximately **59% accuracy** on the FPB test set without learning anything meaningful — this is exactly why accuracy alone is a misleading metric for this task.

**Why class imbalance matters:**

- Classifiers minimising cross-entropy loss will naturally bias toward majority classes
- Minority classes (Fear: 7 test samples, Joy: 9 test samples) are nearly invisible to a model that ignores imbalance
- `class_weight='balanced'` in Logistic Regression and SVM rescales loss contributions inversely proportional to class frequency
- **F1 (macro)** — unweighted average across all classes — is used as the primary metric because it penalises majority-class bias

**FPB imbalance at a glance:**

| Class | Train | Val | Test |
|-------|-------|-----|------|
| Neutral | 2,015 (59.4%) | 432 (59.4%) | 432 (59.4%) |
| Optimism | 912 (26.9%) | 195 (26.8%) | 196 (27.0%) |
| Sadness | 391 (11.5%) | 84 (11.6%) | 83 (11.4%) |
| Joy | 42 (1.2%) | 9 (1.2%) | 9 (1.2%) |
| Fear | 32 (0.9%) | 7 (1.0%) | 7 (1.0%) |

The splits are proportional, confirming stratified sampling was used. Fear and Joy together account for only **2.1% of all FPB data**.


In [ ]:
# Alphabetical order matches sklearn's internal class ordering
# (prevents target_names misalignment in classification_report)
LABELS = ['Fear', 'Joy', 'Neutral', 'Optimism', 'Sadness']
COLORS = ['#C44E52', '#55A868', '#4C72B0', '#DD8452', '#8172B2']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_data = [
    (goemotions, 'GoEmotions (General Domain)'),
    (fpb_train,  'FPB Train (Domain-Specific)'),
    (fpb_test,   'FPB Test (Evaluation Target)'),
]

for ax, (df, title) in zip(axes, plot_data):
    counts = df['label'].value_counts().reindex(LABELS, fill_value=0)
    bars = ax.bar(counts.index, counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel('Count')
    ax.set_xlabel('Sentiment Class')
    ax.tick_params(axis='x', rotation=20)
    for bar, (lbl, cnt) in zip(bars, counts.items()):
        pct = 100 * cnt / len(df)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=7.5)

plt.suptitle('Label Distributions Across Datasets', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Class imbalance summary:')
print(f'  FPB Fear training samples:  {(fpb_train["label"]=="Fear").sum()} ({100*(fpb_train["label"]=="Fear").mean():.1f}%)')
print(f'  FPB Joy  training samples:  {(fpb_train["label"]=="Joy").sum()} ({100*(fpb_train["label"]=="Joy").mean():.1f}%)')
print(f'  Naive-Neutral accuracy on FPB test: {100*(fpb_test["label"]=="Neutral").mean():.1f}%  <- naive baseline')

### 3.2 Class Imbalance Ratio

The chart below quantifies the severity of FPB imbalance by plotting each class's frequency relative to Neutral (the most common class, set to 1.0). GoEmotions is shown for comparison — it is far more balanced across five classes, which is why Model A, despite being trained on 13× more data, fails on domain-specific classes.

**Imbalance ratio = class count ÷ Neutral count.** A ratio of 0.02 (Fear in FPB) means there is 1 Fear sample for every 50 Neutral samples.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (df, name) in zip(axes, [(goemotions, 'GoEmotions'), (fpb_train, 'FPB Train')]):
    counts = df['label'].value_counts().reindex(LABELS, fill_value=0)
    ratio  = counts / counts.max()
    bars   = ax.barh(LABELS, ratio.values, color=COLORS, edgecolor='white')
    ax.axvline(1.0, color='black', linewidth=1.2, linestyle='--', label='Balanced (1.0)')
    ax.set_xlim(0, 1.18)
    ax.set_xlabel('Frequency ratio relative to majority class')
    ax.set_title(f'{name} — Imbalance Ratio', fontweight='bold')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, ratio.values):
        ax.text(v + 0.02, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Class Imbalance: Ratio to Majority Class (Neutral)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Top Bigrams per Domain

Bigrams are two-word sequences that TF-IDF uses alongside single words. Comparing the most frequent bigrams in GoEmotions vs FPB makes the vocabulary gap concrete — not just a metric, but visible in the actual words the models learn from.


In [ ]:
from collections import Counter

def top_bigrams(texts, n=15):
    """Return the n most common bigrams in a list of cleaned texts."""
    bigrams = []
    for text in texts:
        if not isinstance(text, str):
            continue
        tokens = text.lower().split()
        bigrams.extend([f'{a} {b}' for a, b in zip(tokens, tokens[1:])])
    return Counter(bigrams).most_common(n)

# Use clean_text if available, otherwise raw text
go_col  = 'clean_text' if 'clean_text' in goemotions.columns else 'text'
fpb_col = 'clean_text' if 'clean_text' in fpb_train.columns  else 'text'

go_bg  = top_bigrams(goemotions[go_col].values)
fpb_bg = top_bigrams(fpb_train[fpb_col].values)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, bigrams, title, color in [
    (axes[0], go_bg,  'GoEmotions — Top 15 Bigrams', '#4C72B0'),
    (axes[1], fpb_bg, 'FPB Train — Top 15 Bigrams',  '#DD8452'),
]:
    words, counts = zip(*bigrams)
    ax.barh(range(len(words)), counts, color=color, alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Top Bigrams: General Domain vs Financial Domain',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Notice: FPB bigrams are financial terms; GoEmotions bigrams are conversational/emotional.')

### 3.4 Text Length Distribution

GoEmotions (Reddit comments) and FPB (financial news sentences) have distinctly different text length profiles. This matters for TF-IDF in two ways:

1. **Longer documents** produce more bigrams and richer token co-occurrences, improving TF-IDF representation quality
2. **Sublinear TF** (`sublinear_tf=True`) dampens the advantage very long documents would otherwise have

FPB sentences average **~22 words** vs GoEmotions' **~13 words** — nearly twice as long. Financial news packs more information per sentence: *'Net sales rose 8.3% to EUR 2.1bn in Q3, driven by strong performance in Nordic markets'* vs *'this made me so happy omg'*.


In [ ]:
for df in [goemotions, fpb_train, fpb_val, fpb_test]:
    df['word_count'] = df['text'].str.split().apply(len)
    df['char_count'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dsets = [(goemotions,'GoEmotions','#4C72B0'),(fpb_train,'FPB Train','#DD8452'),(fpb_test,'FPB Test','#55A868')]

for metric, ax, title in [('word_count', axes[0], 'Word Count'), ('char_count', axes[1], 'Character Count')]:
    for df, name, color in dsets:
        ax.hist(df[metric].clip(upper=df[metric].quantile(0.99)),
                bins=40, alpha=0.55, label=name, color=color, density=True)
    ax.set_title(f'{title} Distribution', fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Text Length Distributions (GoEmotions vs FPB)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Text length statistics (words):')
print(f'  {"Dataset":<15} | {"Mean":>6} | {"Median":>7} | {"Max":>5} | {"Std":>5}')
print('  ' + '-'*47)
for df, name, _ in dsets:
    l = df['word_count']
    print(f'  {name:<15} | {l.mean():>6.1f} | {l.median():>7.0f} | {l.max():>5} | {l.std():>5.1f}')

### 3.5 Text Length by Sentiment Class (FPB)

Different sentiment classes in FPB may have systematically different text lengths. For example, a short sharp announcement (*'Profit warning issued'*) may lean toward Fear or Sadness, while a longer earnings summary (*'Revenue grew 12% year-on-year driven by ...'*) may lean Optimism or Neutral.

Understanding length differences by class helps explain why certain classes are harder to classify: shorter texts give TF-IDF fewer tokens to work with, making the feature vector sparser and classification noisier.


In [ ]:
fpb_all = pd.concat([fpb_train, fpb_val, fpb_test], ignore_index=True)
order = fpb_all.groupby('label')['word_count'].median().reindex(LABELS).sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=fpb_all, x='label', y='word_count', order=order,
            palette=dict(zip(LABELS, COLORS)), ax=ax, width=0.5, fliersize=3)
ax.set_title('FPB — Word Count Distribution by Sentiment Class', fontweight='bold')
ax.set_xlabel('Sentiment Class')
ax.set_ylabel('Word Count')
ax.set_ylim(0, fpb_all['word_count'].quantile(0.98))
plt.tight_layout()
plt.show()

print('Median word count by class (FPB combined):')
for lbl in LABELS:
    med = fpb_all[fpb_all['label']==lbl]['word_count'].median()
    n   = (fpb_all['label']==lbl).sum()
    print(f'  {lbl:12s}: median {med:.0f} words  (n={n})')

### 3.6 Sample Texts by Class

Inspecting raw text samples makes the vocabulary and tone differences between domains concrete. This qualitative step directly motivates why Model A — trained on informal social media — cannot reliably classify formal financial sentiment.

**Vocabulary contrast:**

| GoEmotions style | FPB style |
|-----------------|----------|
| *'omg I love this so much'* | *'operating profit rose 12% to EUR 45m'* |
| *'why does everything have to be so hard'* | *'the company issued a profit warning citing weak demand'* |
| *'honestly terrified for tomorrow'* | *'credit exposure increased due to counterparty risk'* |

TF-IDF weights learned from the left column are nearly useless for classifying the right column.


In [ ]:
print('GoEmotions — sample sentences by class')
print('='*65)
for lbl in LABELS:
    sample = goemotions[goemotions['label']==lbl]['text'].iloc[0]
    print(f'  [{lbl:10}] {sample[:90]}')

print()
print('FPB — sample sentences by class')
print('='*65)
for lbl in LABELS:
    subset = fpb_train[fpb_train['label']==lbl]
    if len(subset) > 0:
        sample = subset['text'].iloc[0]
        print(f'  [{lbl:10}] {sample[:90]}')

### 3.7 Vocabulary Overlap Between Domains

Measuring how much vocabulary the two domains share **directly quantifies the domain gap** as a single interpretable number. Low overlap is the mechanical explanation for why Model A underperforms: the TF-IDF feature weights it learned from GoEmotions are largely irrelevant to financial text classification.

**Interpretation:**

- Tokens appearing in both domains (e.g. *'good'*, *'growth'*, *'loss'*) have consistent signal
- Tokens exclusive to GoEmotions (e.g. *'omg'*, *'adorable'*, *'wholesome'*) waste feature capacity when applied to FPB
- Tokens exclusive to FPB (e.g. *'amortisation'*, *'turnover'*, *'ebitda'*) are entirely invisible to Model A

This overlap analysis provides the theoretical underpinning for the empirical result we observe: Model B outperforms Model A by ~30 pp macro F1 using 13× fewer training samples.


In [ ]:
def get_vocab(texts, min_freq=3):
    tokens = []
    for t in texts:
        if isinstance(t, str):
            tokens.extend(t.lower().split())
    return set(w for w, c in Counter(tokens).items() if c >= min_freq)

vocab_go  = get_vocab(goemotions['text'])
vocab_fpb = get_vocab(pd.concat([fpb_train, fpb_val])['text'])
overlap   = vocab_go & vocab_fpb
only_go   = vocab_go  - vocab_fpb
only_fpb  = vocab_fpb - vocab_go

print('Vocabulary analysis (tokens appearing >= 3 times)')
print('='*52)
print(f'  GoEmotions vocab size:    {len(vocab_go):>7,}')
print(f'  FPB vocab size:           {len(vocab_fpb):>7,}')
print(f'  Shared tokens:            {len(overlap):>7,}  ({100*len(overlap)/len(vocab_fpb):.1f}% of FPB vocab)')
print(f'  GoEmotions-only tokens:   {len(only_go):>7,}')
print(f'  FPB-only tokens:          {len(only_fpb):>7,}')
print()

fig, ax = plt.subplots(figsize=(9, 3))
categories = ['GoEmotions only', 'Shared', 'FPB only']
values     = [len(only_go), len(overlap), len(only_fpb)]
bar_c      = ['#4C72B0', '#8172B2', '#DD8452']
bars = ax.barh(categories, values, color=bar_c, edgecolor='white')
for bar, v in zip(bars, values):
    ax.text(v + 200, bar.get_y() + bar.get_height()/2,
            f'{v:,}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Number of unique tokens (min freq = 3)')
ax.set_title('Vocabulary Overlap: GoEmotions vs FPB', fontweight='bold')
ax.set_xlim(0, max(values)*1.15)
plt.tight_layout()
plt.show()

print('FPB-exclusive financial tokens (sample):')
print('  ' + ', '.join(sorted(list(only_fpb))[:40]))

### 3.8 Domain-Unique Vocabulary — Top Tokens Exclusive to Each Domain

Beyond the overlap percentage, it's useful to see **which specific tokens appear only in one domain**. These exclusive tokens are the ones TF-IDF weights from GoEmotions that have zero relevance to FPB — and vice versa. Model A can never learn the financial-exclusive tokens; Model B never sees the general-exclusive ones.


In [ ]:
def get_token_counts(texts, min_freq=5):
    """Return a Counter of tokens appearing at least min_freq times."""
    counts = Counter()
    for t in texts:
        if isinstance(t, str):
            counts.update(t.split())
    return Counter({k: v for k, v in counts.items() if v >= min_freq})

from collections import Counter
go_counts  = get_token_counts(goemotions['clean_text'] if 'clean_text' in goemotions.columns else goemotions['text'])
fpb_counts = get_token_counts(fpb_train['clean_text']  if 'clean_text' in fpb_train.columns  else fpb_train['text'])

go_only  = {k: v for k, v in go_counts.items()  if k not in fpb_counts}
fpb_only = {k: v for k, v in fpb_counts.items() if k not in go_counts}

top_go_only  = sorted(go_only.items(),  key=lambda x: -x[1])[:20]
top_fpb_only = sorted(fpb_only.items(), key=lambda x: -x[1])[:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, items, title, color in [
    (axes[0], top_go_only,  'GoEmotions-Only Tokens (not in FPB)', '#4C72B0'),
    (axes[1], top_fpb_only, 'FPB-Only Tokens (not in GoEmotions)', '#DD8452'),
]:
    words, counts = zip(*items)
    ax.barh(range(len(words)), counts, color=color, alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency in source dataset')

plt.suptitle('Vocabulary Exclusive to Each Domain (min frequency = 5)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'GoEmotions-exclusive tokens (freq≥5): {len(go_only):,}')
print(f'FPB-exclusive tokens (freq≥5):        {len(fpb_only):,}')
print('\nThese exclusive tokens are the direct cause of Model A\'s failure on FPB:')
print('TF-IDF features learned from GoEmotions are nearly useless for financial sentiment.')

## 5. Prepare Model A / B / C Training Sets

The three training sets correspond directly to the A/B/C experimental framework.
The test set is always FPB Test — **identical across all nine experiments** to ensure fair comparison.

| Variable | Dataset | Samples | Domain |
|----------|---------|---------|--------|
| X_A, y_A | GoEmotions | 43,404 | General (Reddit) |
| X_B, y_B | FPB Train | 3,392 | Financial news |
| X_C, y_C | GoEmotions + FPB Train | 46,796 | Mixed |
| X_test, y_test | FPB Test | 727 | Financial news (evaluation only) |

In [ ]:
# Model A — General domain only
X_A = goemotions['clean_text'].values
y_A = goemotions['label'].values

# Model B — Domain only
X_B = fpb_train['clean_text'].values
y_B = fpb_train['label'].values

# Model C — Mixed
mixed_df = pd.concat([goemotions, fpb_train], ignore_index=True)
X_C      = mixed_df['clean_text'].values
y_C      = mixed_df['label'].values

# Fixed test set
X_test = fpb_test['clean_text'].values
y_test = fpb_test['label'].values

print('Training set sizes:')
print(f'  Model A (General): {len(X_A):>6,} samples')
print(f'  Model B (Domain):  {len(X_B):>6,} samples')
print(f'  Model C (Mixed):   {len(X_C):>6,} samples')
print(f'  Test set (fixed):  {len(X_test):>6,} samples')

# Visualise composition
fig, ax = plt.subplots(figsize=(9, 3.5))
cond_labels = ['Model A\n(General)', 'Model B\n(Domain)', 'Model C\n(Mixed)']
go_sizes    = [len(X_A), 0, len(goemotions)]
fpb_sizes   = [0, len(X_B), len(fpb_train)]
x = range(3)
b1 = ax.bar(x, go_sizes,  label='GoEmotions', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x, fpb_sizes, bottom=go_sizes, label='FPB Train', color='#DD8452', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(cond_labels)
ax.set_ylabel('Training Samples')
ax.set_title('Training Set Composition per Model Condition', fontweight='bold')
ax.legend()
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + h/2,
                f'{h:,}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 Training Set Composition

Visualising the composition of each training set makes the experimental design concrete. The stacked bar below shows how many samples from each source dataset and class make up Models A, B, and C.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: total training sizes
ax = axes[0]
names  = ['Model A\n(General)', 'Model B\n(Domain)', 'Model C\n(Mixed)']
sizes  = [len(X_A), len(X_B), len(X_C)]
colors_bar = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(names, sizes, color=colors_bar, alpha=0.85, edgecolor='white')
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{s:,}', ha='center', fontweight='bold', fontsize=11)
ax.set_ylabel('Training samples')
ax.set_title('Total Training Set Size per Condition', fontweight='bold')
ax.set_ylim(0, max(sizes) * 1.12)

# Right: class distribution within each training set
ax2 = axes[1]
cmap = ['#C44E52', '#55A868', '#4C72B0', '#DD8452', '#8172B2']
train_dfs = [
    ('A', goemotions),
    ('B', fpb_train),
    ('C', pd.concat([goemotions, fpb_train], ignore_index=True)),
]
x = np.arange(len(train_dfs))
bottoms = np.zeros(3)
for j, lbl in enumerate(LABELS):
    vals = [100 * (df['label'] == lbl).sum() / len(df) for _, df in train_dfs]
    ax2.bar(x, vals, bottom=bottoms, label=lbl, color=cmap[j], alpha=0.85)
    bottoms += np.array(vals)

ax2.set_xticks(x)
ax2.set_xticklabels(['A (General)', 'B (Domain)', 'C (Mixed)'])
ax2.set_ylabel('% of training set')
ax2.set_title('Class Distribution per Training Condition', fontweight='bold')
ax2.legend(loc='upper right', fontsize=8)
ax2.set_ylim(0, 110)

plt.suptitle('Training Set Overview — Models A / B / C',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Build TF-IDF Pipelines

### Why TF-IDF?

TF-IDF (Term Frequency — Inverse Document Frequency) is the standard feature representation for traditional text classification. It assigns each token a weight that reflects how important it is to a particular document relative to the whole corpus.

Common words like *"the"* appear in almost every document (high document frequency) and get low weight. Rare but informative words like *"write-down"* get high weight.

### TF-IDF Configuration

| Parameter | Value | Reason |
|-----------|-------|--------|
| `ngram_range` | (1, 2) | Unigrams and bigrams — bigrams like *"net loss"* carry meaning single words cannot |
| `max_features` | 50,000 | Wide vocabulary without excessive noise |
| `sublinear_tf` | True | Applies log(1+TF) to dampen very frequent tokens |
| `min_df` | 2 | Ignores tokens seen in only one document |

### Classifiers

| Classifier | Key setting | Notes |
|------------|-------------|-------|
| Logistic Regression | `class_weight='balanced'`, `max_iter=1000` | Strong interpretable baseline |
| Linear SVM | `class_weight='balanced'`, `C=0.5`, `max_iter=2000` | Often best in high-dimensional sparse text |
| Naïve Bayes (ComplementNB) | `alpha=0.3` | Complement variant handles imbalance better than MultinomialNB |

Each experiment uses **fresh pipeline instances** — no state is shared between runs.

Three classifiers × three training conditions = **9 experiments total**.

In [ ]:
TFIDF_PARAMS = dict(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
    analyzer='word',
)


def make_pipelines():
    """Return fresh instances of all three classifier pipelines."""
    return {
        'Logistic Regression': Pipeline([
            ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
            ('clf',   LogisticRegression(
                C=1.0, max_iter=1000,
                class_weight='balanced',
                solver='lbfgs', random_state=SEED
            ))
        ]),
        'Linear SVM': Pipeline([
            ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
            ('clf',   LinearSVC(
                C=0.5, max_iter=2000,
                class_weight='balanced', random_state=SEED
            ))
        ]),
        'Naive Bayes': Pipeline([
            ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
            ('clf',   ComplementNB(alpha=0.3))
        ]),
    }


print('Pipelines defined:')
for name in make_pipelines():
    print(f'  - {name}')

## 7. Train & Evaluate — All 9 Experiments

Each pipeline is fitted on its training set then evaluated on the fixed FPB test set. Results are stored in a nested dictionary (`results[condition][classifier]`) for later comparison.

**Key implementation notes:**

- Each call to `make_pipelines()` returns **fresh instances** — no fitted state is shared between experiments
- The TF-IDF vectoriser inside each pipeline is fitted only on that experiment's training data, preventing data leakage
- `evaluate_pipeline()` returns both the metrics dict and the raw `y_pred` array, needed for later error analysis
- The same `X_test` / `y_test` arrays are used in every call — the FPB test set never changes

This produces a fully controlled experiment: **identical architecture, identical test set, only training data varies**.


In [ ]:
def evaluate_pipeline(pipeline, X_train, y_train, X_test, y_test):
    """Fit pipeline, predict, and return a dictionary of evaluation metrics."""
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    return {
        'accuracy':    accuracy_score(y_test, y_pred),
        'f1_macro':    f1_score(y_test, y_pred, average='macro',    zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'precision':   precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall':      recall_score(y_test, y_pred, average='macro',    zero_division=0),
        'y_pred':      y_pred,
        'pipeline':    pipeline,
    }


training_sets = {
    'A (General Only)': (X_A, y_A),
    'B (Domain Only)':  (X_B, y_B),
    'C (Mixed)':        (X_C, y_C),
}

results = {}

for condition, (X_train, y_train) in training_sets.items():
    print(f'\n{"="*62}')
    print(f'  MODEL {condition}  —  training on {len(X_train):,} samples')
    print(f'{"="*62}')
    results[condition] = {}

    for clf_name, pipeline in make_pipelines().items():
        m = evaluate_pipeline(pipeline, X_train, y_train, X_test, y_test)
        results[condition][clf_name] = m
        print(f'  {clf_name:22s} | Acc: {m["accuracy"]:.3f} | '
              f'F1-macro: {m["f1_macro"]:.3f} | F1-weighted: {m["f1_weighted"]:.3f}')

print('\nAll 9 experiments complete!')

### 7.1 Validation Set Evaluation — Model B

The FPB validation set (`fpb_val.csv`, 727 samples) provides an independent check of Model B's generalisation. Comparing validation and test performance reveals whether any overfitting occurred and confirms the test results are not an anomaly.

**Expected:** Val and test F1 should be close — both are held-out FPB splits with the same distribution.


In [ ]:
print('Model B — Validation vs Test F1 (macro)\n')
print(f'{"Classifier":<25} {"Val F1":>10} {"Test F1":>10} {"Gap":>8}')
print('-' * 58)

X_val = fpb_val['clean_text'].values
y_val = fpb_val['label'].values

for clf_name, pipeline in make_pipelines().items():
    pipeline.fit(X_B, y_B)
    f1_val  = f1_score(y_val,  pipeline.predict(X_val),  average='macro', zero_division=0)
    f1_test = f1_score(y_test, pipeline.predict(X_test), average='macro', zero_division=0)
    gap = f1_test - f1_val
    print(f'{clf_name:<25} {f1_val:>10.4f} {f1_test:>10.4f} {gap:>+8.4f}')

print('\nSmall gap (< 0.05) confirms models generalise and are not overfitting FPB train data.')

## 8. Consolidated Results Table

All metrics are computed on the same FPB test set.

**Reading the table:**
- F1 (macro) is the primary diagnostic for minority-class performance
- F1 (weighted) reflects real-world utility given the imbalanced test set
- Green shading indicates higher values

In [ ]:
conditions  = list(training_sets.keys())
classifiers = ['Logistic Regression', 'Linear SVM', 'Naive Bayes']

rows = []
for condition in conditions:
    for clf in classifiers:
        m = results[condition][clf]
        rows.append({
            'Model Condition':   f'Model {condition}',
            'Classifier':        clf,
            'Accuracy':          round(m['accuracy'],    4),
            'Precision (macro)': round(m['precision'],   4),
            'Recall (macro)':    round(m['recall'],      4),
            'F1 (macro)':        round(m['f1_macro'],    4),
            'F1 (weighted)':     round(m['f1_weighted'], 4),
        })

results_df = pd.DataFrame(rows)

(
    results_df.style
    .background_gradient(cmap='YlGn', subset=['F1 (macro)', 'F1 (weighted)', 'Accuracy'])
    .format(precision=4)
    .set_caption('Baseline ML Performance on FPB Test Set — All 9 Experiments')
)

## 9. Metric Comparison — Bar Charts

Three metrics are visualised side by side to give a complete performance picture:

- **Accuracy** — straightforward but misleading at 59% Neutral imbalance
- **F1 (Macro)** — the most important metric for this task; catches minority-class failures
- **F1 (Weighted)** — reflects real-world utility; more lenient on minority classes

**What to expect from the charts:**

- Model A bars should be clearly shorter than B and C
- All three classifiers within each condition should be clustered close together — the gap between conditions is larger than the gap between classifiers, confirming data drives performance
- Naive Bayes may show more variance because it lacks `class_weight` support


In [ ]:
bar_colors      = ['#4C72B0', '#DD8452', '#55A868']
metrics_to_plot = [('accuracy', 'Accuracy'), ('f1_macro', 'F1 (Macro)'), ('f1_weighted', 'F1 (Weighted)')]

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
x     = np.arange(len(conditions))
width = 0.25

for ax, (metric, label) in zip(axes, metrics_to_plot):
    for i, clf in enumerate(classifiers):
        values = [results[c][clf][metric] for c in conditions]
        bars   = ax.bar(x + i * width, values, width,
                        label=clf, color=bar_colors[i], alpha=0.85)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x + width)
    ax.set_xticklabels(['A\n(General)', 'B\n(Domain)', 'C\n(Mixed)'])
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)

plt.suptitle('Baseline Performance: Model A vs B vs C — Evaluated on FPB Test Set',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 9.1 Radar Chart — Multi-Metric View (Best Classifier per Condition)

A radar chart shows all five metrics simultaneously for the best classifier in each condition, making trade-offs between precision, recall, and F1 immediately visible.

In [ ]:
def best_clf(condition):
    return max(results[condition], key=lambda c: results[condition][c]['f1_macro'])

radar_metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'precision', 'recall']
radar_labels  = ['Accuracy', 'F1\nMacro', 'F1\nWeighted', 'Precision\nMacro', 'Recall\nMacro']
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
radar_colors = ['#4C72B0', '#DD8452', '#55A868']

for i, condition in enumerate(conditions):
    b      = best_clf(condition)
    values = [results[condition][b][m] for m in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2,
            label=f'Model {condition[:1]} ({b})', color=radar_colors[i])
    ax.fill(angles, values, alpha=0.12, color=radar_colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], size=8)
ax.set_title('Multi-Metric Radar — Best Classifier per Condition', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

### 9.2 Precision vs Recall Scatter — Per Class and Condition

A precision–recall scatter plot shows the trade-off each classifier makes per class. Points in the top-right are ideal (high precision and recall). Points on axes indicate the model almost never predicts that class (high precision, zero recall) or predicts it constantly and wrongly (low precision, high recall).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
markers  = ['o', 's', '^']
clf_list = list(results[conditions[0]].keys())
colors_cls = ['#C44E52', '#55A868', '#4C72B0', '#DD8452', '#8172B2']

for ax, condition in zip(axes, conditions):
    for ci, clf_name in enumerate(clf_list):
        y_pred = results[condition][clf_name]['y_pred']
        report = classification_report(
            y_test, y_pred, labels=LABELS, target_names=LABELS,
            zero_division=0, output_dict=True
        )
        for li, lbl in enumerate(LABELS):
            p = report[lbl]['precision']
            r = report[lbl]['recall']
            ax.scatter(r, p, color=colors_cls[li],
                       marker=markers[ci], s=90, alpha=0.85,
                       label=f'{lbl} ({clf_name[:2]})' if condition == conditions[0] else '')
            if ci == 0:  # label the class name once
                ax.annotate(lbl[:3], (r, p), textcoords='offset points',
                            xytext=(5, 3), fontsize=7, color=colors_cls[li])
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'Model {condition[:1]}', fontweight='bold')
    ax.plot([0,1],[0,1], 'k--', alpha=0.2, linewidth=0.8)  # diagonal reference
    ax.axhline(0.5, color='grey', linewidth=0.5, linestyle=':')
    ax.axvline(0.5, color='grey', linewidth=0.5, linestyle=':')

# Custom legend
from matplotlib.lines import Line2D
legend_elements = (
    [Line2D([0],[0], marker=m, color='grey', linestyle='None', markersize=8, label=c[:10])
     for m, c in zip(markers, clf_list)] +
    [Line2D([0],[0], marker='o', color=col, linestyle='None', markersize=8, label=lbl)
     for col, lbl in zip(colors_cls, LABELS)]
)
fig.legend(handles=legend_elements, loc='lower center', ncol=8,
           bbox_to_anchor=(0.5, -0.12), fontsize=8)

plt.suptitle('Precision vs Recall per Class — All Three Conditions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Confusion Matrices — Best Classifier per Condition

Confusion matrices reveal **what the model is actually confusing**, not just how often it fails. Each cell `[i][j]` shows how many true-class-`i` samples were predicted as class `j`.

- **Diagonal** (top-left to bottom-right) = correct predictions — we want high values here
- **Off-diagonal** = errors — we want these as close to zero as possible
- **Rows** = true labels; **Columns** = predicted labels

**Common patterns to look for:**

| Pattern | Meaning |
|---------|----------|
| Entire row → Neutral column | Model predicts Neutral for everything in that class |
| Fear row entirely empty | Model never predicts Fear (extreme imbalance effect) |
| Optimism confused with Neutral | Mild positive financial language is ambiguous |
| Sadness confused with Neutral | Subtle financial loss language can appear neutral |

The best classifier (by macro F1) is selected for each condition independently.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, condition in zip(axes, conditions):
    b      = best_clf(condition)
    y_pred = results[condition][b]['y_pred']
    cm     = confusion_matrix(y_test, y_pred, labels=LABELS)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    f1 = results[condition][b]['f1_macro']
    ax.set_title(f'Model {condition[:1]} — {b}\nF1-macro: {f1:.3f}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

plt.suptitle('Confusion Matrices — Best Classifier per Condition',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 10.1 Normalised Confusion Matrices (Recall View)

Normalising by true-class totals converts raw counts to **recall per class** — the fraction of each true class that was correctly identified. This makes cross-class comparison fair on imbalanced data: a model that correctly identifies 1 out of 7 Fear samples (recall = 0.14) and 381 out of 432 Neutral samples (recall = 0.88) shows this disparity clearly.

**Why normalise?** Raw confusion matrices are dominated by Neutral (432 samples) vs Fear (7 samples). Normalised matrices weight all classes equally by row, making minority-class patterns visible.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, condition in zip(axes, conditions):
    b      = best_clf(condition)
    y_pred = results[condition][b]['y_pred']
    cm     = confusion_matrix(y_test, y_pred, labels=LABELS, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS,
                ax=ax, vmin=0, vmax=1, cbar=False)
    ax.set_title(f'Model {condition[:1]} — {best_clf(condition)}\n(row-normalised: recall per class)',
                 fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

plt.suptitle('Normalised Confusion Matrices — Recall View',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 10.2 Misclassification Summary — Where Errors Go

The confusion matrix shows counts; this table shows **where errors land** as a percentage of each class's test samples. For example, '72% of Optimism errors → predicted Neutral' tells us the model conflates financial optimism with neutral statements.


In [ ]:
print('Misclassification breakdown — best classifier per condition\n')

for condition in conditions:
    b = best_clf(condition)
    y_pred = results[condition][b]['y_pred']
    cm = confusion_matrix(y_test, y_pred, labels=LABELS)
    print(f'Model {condition[:1]} ({b})')
    print(f'  {"True Class":<12}  {"Most common error (predicted as)"}' )
    print('  ' + '-'*55)
    for ri, true_lbl in enumerate(LABELS):
        row = cm[ri].copy()
        total = row.sum()
        correct = row[ri]
        row[ri] = 0  # mask correct predictions
        if row.sum() == 0:
            print(f'  {true_lbl:<12}: all correct (n={total})')
        else:
            top_err_idx = row.argmax()
            top_err_lbl = LABELS[top_err_idx]
            top_err_pct = 100 * row[top_err_idx] / total
            print(f'  {true_lbl:<12}: → {top_err_lbl:<12} ({top_err_pct:.0f}% of {total} samples)')
    print()

## 11. Per-Class Classification Reports

Full precision, recall and F1 for every class and every best-classifier experiment. This is the most granular evaluation output — it contains the full diagnostic information for the technical report.

**Reading the report:**

| Metric | Measures | Good when |
|--------|----------|----------|
| Precision | Of all predicted-as-X, how many were truly X? | High = few false alarms |
| Recall | Of all true-X, how many were predicted as X? | High = few misses |
| F1 | Harmonic mean of precision and recall | High = both precision and recall are good |
| Support | Number of true instances in test set | Context for interpreting F1 |

**What to look for:**

- **Fear and Joy** — minority classes with fewest training examples. Near-zero F1 in Model A reveals the domain gap directly. The Fear row often shows P=1.0, R≈0.14 in Model B — meaning when the model predicts Fear it is always right, but it almost never dares to predict Fear at all.
- **Neutral** — the majority class. High recall but potentially lower precision (over-predicts Neutral when unsure).
- **Optimism and Sadness** — large F1 gains expected moving from A to B, since these financial concepts have strong domain vocabulary.


In [ ]:
for condition in conditions:
    b      = best_clf(condition)
    y_pred = results[condition][b]['y_pred']
    print(f'\n{"="*62}')
    print(f'  Model {condition}  —  Best Classifier: {b}')
    print('=' * 62)
    print(classification_report(y_test, y_pred, labels=LABELS,
                                target_names=LABELS, zero_division=0))

## 12. Per-Class F1 Heatmap — All 9 Experiments

Each row is one experiment (condition + classifier), each column is one sentiment class.

**Green** = high F1 | **Red** = low or zero F1

Key patterns to identify:
- Entire rows that are red → a classifier that fails across all classes
- Specific columns that are red → universally hard classes (Fear, Joy)
- The improvement gradient from Model A rows → Model B rows

In [ ]:
heatmap_data, row_labels = [], []

for condition in conditions:
    for clf in classifiers:
        y_pred = results[condition][clf]['y_pred']
        report = classification_report(
            y_test, y_pred, labels=LABELS, target_names=LABELS,
            zero_division=0, output_dict=True
        )
        heatmap_data.append([report[lbl]['f1-score'] for lbl in LABELS])
        row_labels.append(f'Model {condition[:1]} — {clf}')

hmap_df = pd.DataFrame(heatmap_data, index=row_labels, columns=LABELS)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(hmap_df, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Per-Class F1 Scores — All 9 Experiments', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment Class')
plt.tight_layout()
plt.show()

### 12.1 Per-Class F1 — Side-by-Side Bar Chart (Best Classifier per Condition)

This chart directly shows how each individual sentiment class improves (or degrades) as training data changes from A → B → C. Unlike the heatmap, the bar chart makes magnitude differences easy to read at a glance.

**What to look for:**

- **Neutral** — likely high across all conditions, but may vary as class-weight calibration changes
- **Optimism and Sadness** — large jump expected from A to B (domain vocabulary kicks in)
- **Fear and Joy** — remain low even in B and C due to extreme data scarcity
- **Model C** — sits between A and B for most classes, reflecting the 13:1 dilution effect


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x     = np.arange(len(LABELS))
width = 0.25
cond_colors = ['#4C72B0', '#DD8452', '#55A868']

for i, condition in enumerate(conditions):
    b      = best_clf(condition)
    y_pred = results[condition][b]['y_pred']
    report = classification_report(
        y_test, y_pred, labels=LABELS, target_names=LABELS,
        zero_division=0, output_dict=True
    )
    f1s  = [report[lbl]['f1-score'] for lbl in LABELS]
    bars = ax.bar(x + i*width, f1s, width,
                  label=f'Model {condition[:1]} ({b})',
                  color=cond_colors[i], alpha=0.85)
    for bar, v in zip(bars, f1s):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width)
ax.set_xticklabels(LABELS, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('F1-Score')
ax.set_title('Per-Class F1: Model A vs B vs C (Best Classifier per Condition)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 12.2 F1 Change Heatmap — Δ(A→B) and Δ(B→C)

Rather than showing absolute F1, this heatmap shows the **change** in per-class F1 as training data is adapted: first from general-only (A) to domain-only (B), then from domain-only (B) to mixed (C). Green = improvement, red = degradation.


In [ ]:
best_per = {c: best_clf(c) for c in conditions}

def per_class_f1s(condition):
    b = best_per[condition]
    y_pred = results[condition][b]['y_pred']
    rep = classification_report(y_test, y_pred, labels=LABELS, target_names=LABELS,
                                zero_division=0, output_dict=True)
    return np.array([rep[lbl]['f1-score'] for lbl in LABELS])

f1_A = per_class_f1s('A (General Only)')
f1_B = per_class_f1s('B (Domain Only)')
f1_C = per_class_f1s('C (Mixed)')

delta_AB = f1_B - f1_A   # improvement from A to B
delta_BC = f1_C - f1_B   # improvement from B to C

delta_df = pd.DataFrame(
    [delta_AB, delta_BC],
    index=['Δ A → B\n(General to Domain)', 'Δ B → C\n(Domain to Mixed)'],
    columns=LABELS
)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(
    delta_df, annot=True, fmt='+.3f', cmap='RdYlGn',
    center=0, vmin=-0.4, vmax=0.6,
    linewidths=0.5, ax=ax, annot_kws={'size': 11, 'weight': 'bold'}
)
ax.set_title('Per-Class F1 Change (Δ) Between Training Conditions — Best Classifier',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Sentiment Class')
plt.tight_layout()
plt.show()

print('Δ A→B: the gain from moving to domain-specific training')
print('Δ B→C: the additional gain (or cost) from adding general data')
print(f'\nLargest gain A→B: {LABELS[delta_AB.argmax()]} (+{delta_AB.max():.3f})')
print(f'Smallest gain A→B: {LABELS[delta_AB.argmin()]} ({delta_AB.min():+.3f})')

## 13. TF-IDF Feature Analysis — Vocabulary Gap

Inspecting the highest-weighted TF-IDF features per class for Logistic Regression is one of the most interpretable diagnostic steps in classical NLP.

**What we expect:**
- **Model A** top features: social-media emotion words (*"love"*, *"hate"*, *"amazing"*, *"omg"*)
- **Model B** top features: financial terminology (*"profit"*, *"net loss"*, *"operating"*, *"revenue"*)

This vocabulary gap is the mechanical explanation for Model A’s low performance on FPB.

In [ ]:
def show_top_features(condition, clf_name='Logistic Regression', top_n=10):
    pipeline    = results[condition][clf_name]['pipeline']
    vectorizer  = pipeline.named_steps['tfidf']
    classifier  = pipeline.named_steps['clf']
    feat_names  = vectorizer.get_feature_names_out()

    print(f'\nTop features — Model {condition} ({clf_name})')
    print('-' * 65)
    for i, cls in enumerate(classifier.classes_):
        top_idx = np.argsort(classifier.coef_[i])[-top_n:][::-1]
        print(f'  {cls:12s}: {", ".join(feat_names[top_idx])}')


show_top_features('A (General Only)')
show_top_features('B (Domain Only)')
show_top_features('C (Mixed)')

### 13.1 Top Feature Weights — Visual Comparison (Model A vs Model B)

Plotting the top-weighted tokens for each class makes the vocabulary gap visually immediate. Each panel shows the 6 tokens with the highest Logistic Regression coefficient for that class — the words that most strongly push the model toward predicting that label.

**Expected contrasts:**

| Class | Model A top tokens (GoEmotions) | Model B top tokens (FPB) |
|-------|--------------------------------|-------------------------|
| Optimism | *happy*, *love*, *great*, *amazing* | *growth*, *increase*, *strong*, *outlook* |
| Sadness | *sad*, *cry*, *depressed*, *hurt* | *loss*, *decline*, *write down*, *fell* |
| Fear | *scared*, *afraid*, *worried*, *panic* | *risk*, *uncertainty*, *exposure*, *concern* |
| Neutral | *just*, *like*, *really*, *think* | *said*, *company*, *quarter*, *reported* |

The visual confirms why Model A fails: its coefficients point at social-media vocabulary that has zero discriminative power on financial news.


In [ ]:
fig, axes = plt.subplots(2, len(LABELS), figsize=(18, 8))
top_n = 6

for row_idx, condition in enumerate(['A (General Only)', 'B (Domain Only)']):
    pipeline   = results[condition]['Logistic Regression']['pipeline']
    vectorizer = pipeline.named_steps['tfidf']
    classifier = pipeline.named_steps['clf']
    feat_names = vectorizer.get_feature_names_out()

    for col_idx, cls in enumerate(LABELS):
        ax = axes[row_idx][col_idx]
        cls_idx = list(classifier.classes_).index(cls)
        top_idx = np.argsort(classifier.coef_[cls_idx])[-top_n:]
        words   = feat_names[top_idx]
        weights = classifier.coef_[cls_idx][top_idx]

        ax.barh(range(top_n), weights, color=COLORS[col_idx], alpha=0.8)
        ax.set_yticks(range(top_n))
        ax.set_yticklabels(words, fontsize=8)
        ax.set_title(f'{cls}' if row_idx == 0 else '', fontweight='bold', fontsize=9)
        ax.set_xlabel('Weight', fontsize=7)
        if col_idx == 0:
            ax.set_ylabel(f'Model {condition[:1]}', fontsize=9, fontweight='bold')

plt.suptitle('Top TF-IDF Features per Class: Model A (General) vs Model B (Domain)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 14. Error Analysis

Error analysis goes beyond aggregate metrics to understand **specific prediction mistakes**. Rather than asking 'how often did the model fail?', it asks 'which cases failed, and why?'

**Experimental design:**

We use the best classifier (by macro F1) from each condition and compare their predictions on every test sample. Four categories emerge:

| Category | Count | Interpretation |
|----------|-------|----------------|
| All 3 correct | — | Easy examples all models handle |
| All 3 wrong | — | Genuinely ambiguous examples beyond TF-IDF capacity |
| **Only A wrong** | — | **Domain gap** — B and C know the financial vocabulary, A doesn't |
| Only B wrong | — | **Data scarcity** — A and C have more general pattern coverage |

The 'Only A wrong' count is the most theoretically important: it directly measures the number of test samples that domain adaptation successfully rescues.


In [ ]:
pred_A = results['A (General Only)'][best_clf('A (General Only)')]['y_pred']
pred_B = results['B (Domain Only)'][best_clf('B (Domain Only)')]['y_pred']
pred_C = results['C (Mixed)'][best_clf('C (Mixed)')]['y_pred']

error_df           = fpb_test[['text', 'label']].copy()
error_df['pred_A'] = pred_A
error_df['pred_B'] = pred_B
error_df['pred_C'] = pred_C

a_wrong_bc_right = error_df[
    (error_df['pred_A'] != error_df['label']) &
    (error_df['pred_B'] == error_df['label']) &
    (error_df['pred_C'] == error_df['label'])
]

print(f'Cases where Model A is wrong but both B and C are correct: {len(a_wrong_bc_right)}')
print('These sentences expose the domain vocabulary gap.\n')
for _, row in a_wrong_bc_right.head(5).iterrows():
    print(f'  Text:        {row["text"][:105]}')
    print(f'  True label:  {row["label"]:12s}  A predicted: {row["pred_A"]}')
    print()

In [ ]:
total = len(error_df)
stats = {
    'All 3 correct':             ((error_df['pred_A']==error_df['label']) & (error_df['pred_B']==error_df['label']) & (error_df['pred_C']==error_df['label'])).sum(),
    'All 3 wrong':               ((error_df['pred_A']!=error_df['label']) & (error_df['pred_B']!=error_df['label']) & (error_df['pred_C']!=error_df['label'])).sum(),
    'Only A wrong (domain gap)': ((error_df['pred_A']!=error_df['label']) & (error_df['pred_B']==error_df['label']) & (error_df['pred_C']==error_df['label'])).sum(),
    'Only B wrong (data gap)':   ((error_df['pred_A']==error_df['label']) & (error_df['pred_B']!=error_df['label']) & (error_df['pred_C']==error_df['label'])).sum(),
}

print('Model Agreement Summary (best classifier per condition)')
print('=' * 58)
for lbl, count in stats.items():
    print(f'  {lbl:35s}: {count:>4}  ({100*count/total:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
labels_p = list(stats.keys())
counts_p = list(stats.values())
colors_p = ['#55A868', '#C44E52', '#DD8452', '#4C72B0']
bars = ax.barh(labels_p, counts_p, color=colors_p, alpha=0.85, edgecolor='white')
for bar, v in zip(bars, counts_p):
    ax.text(v + 3, bar.get_y() + bar.get_height()/2,
            f'{v}  ({100*v/total:.1f}%)', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Number of Test Samples')
ax.set_title('Prediction Agreement Across Models A / B / C', fontweight='bold')
ax.set_xlim(0, max(counts_p) * 1.3)
plt.tight_layout()
plt.show()

### 14.1 Error Breakdown by True Class

Which classes are most responsible for errors in each model? This reveals which sentiment classes each model struggles with most.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (condition, pred_col) in zip(axes, [
    ('A (General Only)', 'pred_A'),
    ('B (Domain Only)',  'pred_B'),
    ('C (Mixed)',        'pred_C'),
]):
    errors     = error_df[error_df[pred_col] != error_df['label']]
    err_counts = errors['label'].value_counts().reindex(LABELS, fill_value=0)
    bars = ax.bar(LABELS, err_counts.values, color=COLORS, edgecolor='white')
    ax.set_title(f'Model {condition[:1]} — Errors by True Class\n(best: {best_clf(condition)})',
                 fontweight='bold')
    ax.set_ylabel('Misclassified count')
    ax.set_xlabel('True Label')
    ax.tick_params(axis='x', rotation=20)
    ax.set_ylim(0, max(err_counts.values) * 1.25)
    total_err = errors.shape[0]
    for bar, v in zip(bars, err_counts.values):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(v), ha='center', va='bottom', fontsize=9)
    ax.text(0.98, 0.97, f'Total errors: {total_err}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='darkred')

plt.suptitle('Error Breakdown by True Sentiment Class', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 14.2 Prediction Agreement Table — Class-Level Breakdown

The agreement analysis above gives totals. This table breaks down how often the best classifier in each condition agrees with the true label — class by class. It reveals which specific classes drive the 'Only A wrong' effect and which classes all three models still struggle with.


In [ ]:
print('Per-class correct predictions — best classifier per condition\n')
print(f'{"Class":<12} {"n test":>7} {"A correct":>11} {"B correct":>11} {"C correct":>11}')
print('-' * 58)

pred_A = results['A (General Only)'][best_clf('A (General Only)')]['y_pred']
pred_B = results['B (Domain Only)'][best_clf('B (Domain Only)')]['y_pred']
pred_C = results['C (Mixed)'][best_clf('C (Mixed)')]['y_pred']

for lbl in LABELS:
    mask = (np.array(y_test) == lbl)
    n = mask.sum()
    a_ok = ((pred_A == y_test) & mask).sum()
    b_ok = ((pred_B == y_test) & mask).sum()
    c_ok = ((pred_C == y_test) & mask).sum()
    print(f'{lbl:<12} {n:>7} {a_ok:>5} ({100*a_ok/n:4.0f}%) '
          f'{b_ok:>5} ({100*b_ok/n:4.0f}%) '
          f'{c_ok:>5} ({100*c_ok/n:4.0f}%)')

total = len(y_test)
ta = (pred_A == y_test).sum(); tb = (pred_B == y_test).sum(); tc = (pred_C == y_test).sum()
print('-' * 58)
print(f'{"TOTAL":<12} {total:>7} {ta:>5} ({100*ta/total:4.0f}%) '
      f'{tb:>5} ({100*tb/total:4.0f}%) '
      f'{tc:>5} ({100*tc/total:4.0f}%)')

## 15. Sample Efficiency — Model B Learning Curve

A learning curve shows how performance scales with the amount of training data. This is especially informative for Model B (domain-only, 3,392 samples):

- If the curve is still rising at full data → **more domain data would help**
- If it has plateaued → **architecture improvements (e.g. BERT) are needed**, not more data

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes = np.linspace(0.1, 1.0, 8)

pipe_lc = Pipeline([
    ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
    ('clf',   LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', solver='lbfgs', random_state=SEED))
])

train_sz, train_scores, val_scores = learning_curve(
    pipe_lc, X_B, y_B,
    train_sizes=train_sizes,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sz, train_mean, 'o-', color='#4C72B0', label='Training F1 (macro)', linewidth=2)
ax.fill_between(train_sz, train_mean-train_std, train_mean+train_std, alpha=0.15, color='#4C72B0')
ax.plot(train_sz, val_mean,   's-', color='#DD8452', label='CV Validation F1 (macro)', linewidth=2)
ax.fill_between(train_sz, val_mean-val_std, val_mean+val_std, alpha=0.15, color='#DD8452')
ax.set_xlabel('Training Set Size (samples)', fontsize=11)
ax.set_ylabel('F1 (Macro)', fontsize=11)
ax.set_title('Learning Curve — Model B (Domain Only, Logistic Regression)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'CV F1 (macro) at full FPB train size: {val_mean[-1]:.3f} +/- {val_std[-1]:.3f}')
print('Training gap (train - val):', round(train_mean[-1] - val_mean[-1], 3), '(higher = more overfitting)')

### 15.1 Train vs Validation F1 — Generalisation Gap (Model B)

The learning curve shows test performance. This complementary plot adds **training F1** alongside validation F1 to reveal the generalisation gap at each data size. A large gap (train >> val) indicates overfitting; a small gap confirms the model is generalising rather than memorising.


In [ ]:
from sklearn.model_selection import learning_curve

lc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
    ('clf', LinearSVC(C=0.5, max_iter=2000, class_weight='balanced', random_state=SEED))
])

train_sizes_abs, train_scores, val_scores = learning_curve(
    lc_pipeline, X_B, y_B,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)
gap        = train_mean - val_mean

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: train vs val F1
ax1.plot(train_sizes_abs, train_mean, 'o-', color='#4C72B0', label='Training F1 (CV mean)')
ax1.fill_between(train_sizes_abs, train_mean-train_std, train_mean+train_std,
                 alpha=0.15, color='#4C72B0')
ax1.plot(train_sizes_abs, val_mean, 's--', color='#DD8452', label='Validation F1 (CV mean)')
ax1.fill_between(train_sizes_abs, val_mean-val_std, val_mean+val_std,
                 alpha=0.15, color='#DD8452')
ax1.set_xlabel('Training samples')
ax1.set_ylabel('Macro F1')
ax1.set_title('Train vs Validation F1 — Model B (Linear SVM)', fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 1)

# Right: generalisation gap
ax2.plot(train_sizes_abs, gap, 'D-', color='#C44E52')
ax2.fill_between(train_sizes_abs, gap-0.01, gap+0.01, alpha=0.2, color='#C44E52')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Training samples')
ax2.set_ylabel('Train F1 − Val F1 (generalisation gap)')
ax2.set_title('Generalisation Gap vs Training Size', fontweight='bold')
ax2.set_ylim(-0.05, 0.5)

plt.suptitle('Model B Generalisation Analysis (5-fold CV)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Gap at minimum training size: {gap[0]:.3f}')
print(f'Gap at full training size:    {gap[-1]:.3f}')
print('A shrinking gap confirms reduced overfitting as training data grows.')

## 16. Best Model Summary Chart

The chart below identifies the best-performing classifier (by macro F1) for each training condition, providing a clean visual summary of the A vs B vs C comparison across all three metrics.

In [ ]:
summary_rows = []
for condition in conditions:
    b = best_clf(condition)
    summary_rows.append({
        'Condition':       f'Model {condition}',
        'Best Classifier': b,
        'F1 (macro)':      results[condition][b]['f1_macro'],
        'F1 (weighted)':   results[condition][b]['f1_weighted'],
        'Accuracy':        results[condition][b]['accuracy'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x  = np.arange(len(summary_df))
w  = 0.25
b1 = ax.bar(x - w,   summary_df['F1 (macro)'],    w, label='F1 (macro)',    color='#4C72B0', alpha=0.85)
b2 = ax.bar(x,       summary_df['F1 (weighted)'],  w, label='F1 (weighted)', color='#8172B2', alpha=0.85)
b3 = ax.bar(x + w,   summary_df['Accuracy'],       w, label='Accuracy',      color='#DD8452', alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['Condition']}\n({r['Best Classifier']})" for _, r in summary_df.iterrows()],
    fontsize=9
)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Best Baseline Performance per Training Condition', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 16.1 Model Card — Best Baseline Configuration

The table below summarises the winning configuration as a concise model card, suitable for inclusion in the technical report. It establishes the target that Transformer models in Tasks 3–5 must exceed.


In [ ]:
conditions  = list(training_sets.keys())
classifiers = ['Logistic Regression', 'Linear SVM', 'Naive Bayes']

best_condition = max(
    [(c, clf) for c in conditions for clf in classifiers],
    key=lambda x: results[x[0]][x[1]]['f1_macro']
)
bc, bclf = best_condition
bm = results[bc][bclf]
y_pred_best = bm['y_pred']
rep = classification_report(y_test, y_pred_best, labels=LABELS, target_names=LABELS,
                             zero_division=0, output_dict=True)

card = {
    'Field': [
        'Model type', 'Classifier', 'Training data', 'Training samples',
        'Test data', 'Test samples', 'Feature extraction',
        'Vocabulary size', 'Accuracy', 'F1 (Macro)', 'F1 (Weighted)',
        'Best class F1', 'Worst class F1', 'Class weighting'
    ],
    'Value': [
        'TF-IDF + ' + bclf,
        bclf,
        bc,
        f'{len(X_B):,} (FPB Train)',
        '../data/processed/fpb_test.csv',
        f'{len(X_test):,}',
        'Unigrams + Bigrams, max_features=50,000, sublinear_tf=True',
        '~10,387 (min_df=2 on FPB corpus)',
        f'{bm["accuracy"]:.4f}',
        f'{bm["f1_macro"]:.4f}',
        f'{bm["f1_weighted"]:.4f}',
        f'{max(rep[l]["f1-score"] for l in LABELS):.4f} (Neutral)',
        f'{min(rep[l]["f1-score"] for l in LABELS):.4f} (Fear)',
        'class_weight=balanced'
    ]
}

card_df = pd.DataFrame(card)
display(card_df.style.hide(axis='index').set_caption(
    f'Best Baseline Model Card — Target for Tasks 3–5'
))
print(f'\nThis is the benchmark all Transformer experiments must exceed.')
print(f'Target for Task 3: macro F1 > {bm["f1_macro"] + 0.10:.2f} (minimum meaningful gain over baseline)')

## 17. Key Findings & Interpretation

### Summary at a Glance

| | Model A — General Only | Model B — Domain Only | Model C — Mixed |
|---|---|---|---|
| **Strength** | Large training signal; broad language patterns | Domain vocabulary; financial semantics | Coverage from both sources |
| **Weakness** | Vocabulary mismatch with financial text | Few samples; severe class imbalance | Still limited by bag-of-words representation |
| **Expected behaviour** | Domain-weak | Domain-aware but unstable | Most balanced |

---

### Finding 1 — Domain Gap (Model A)

Training on GoEmotions does not transfer cleanly to financial text. Words like *"profit"*, *"write-down"*, and *"quarterly results"* carry domain-specific sentiment meaning entirely absent from GoEmotions vocabulary. The model defaults to predicting the majority class (Neutral, 59% of the test set), producing near-zero F1 for minority financial classes. The error analysis confirms this: approximately **23% of test samples** are cases where only Model A fails.

---

### Finding 2 — Data Scarcity (Model B)

FPB train has only 3,392 samples with extreme class imbalance — 42 Joy and 32 Fear training examples. Despite knowing the domain vocabulary, minority-class performance is constrained by insufficient training signal. `class_weight='balanced'` helps but cannot fully compensate.

---

### Finding 3 — Mixed Training (Model C)

At the TF-IDF baseline level, Model C slightly underperforms Model B. The GoEmotions:FPB ratio is approximately 13:1, causing financial-specific vocabulary to be diluted in the combined TF-IDF feature weights. However, Model C substantially outperforms Model A (+27 pp macro F1), confirming that **any domain exposure is better than none**.

This dilution effect is expected to diminish with Transformer-based models, which use attention to selectively weight domain-relevant tokens.

---

### Finding 4 — TF-IDF Limitation

All three conditions share the same fundamental weakness: bag-of-words ignores word order and context. The sentences *"did not lose money"* and *"lost money"* produce nearly identical TF-IDF vectors. This motivates Transformer-based models (Task 3), which capture contextual meaning through self-attention.

---

### These Baselines Serve As

- **Lower bounds** for Transformer model performance in Tasks 3 and 4
- **Experimental evidence** that training data composition matters independently of model architecture
- **Reference benchmarks** for measuring the added value of sequential fine-tuning and LLM prompting

---

> **Core takeaway:** With identical architectures, training data composition significantly changes performance. A ~30 percentage point macro F1 gap exists between identical classifiers trained on different data. This directly answers the research question: **training data choice is as important as model architecture** in domain-adaptive NLP.

## 18. Save Results

Results are exported to CSV for use in the technical report, presentation, and comparison against Transformer models (Task 3) and LLM prompting (Task 5).

In [ ]:
results_df.to_csv('task2_baseline_results.csv', index=False)
print('Results saved to task2_baseline_results.csv')
print()
display(results_df)

## 19. Complete Results Reference

The table below is the authoritative record of all nine baseline experiments. All figures are on the fixed FPB test set (727 samples). This table should be reproduced in the technical report and used as the reference point when comparing against Transformer-based results in Tasks 3–5.


In [ ]:
# Full cross-dataset performance summary
print('Complete Baseline Results — All 9 Experiments on FPB Test Set\n')
print(f'{"Model":<22} {"Classifier":<22} {"Acc":>7} {"F1-mac":>8} {"F1-wgt":>8}')
print('=' * 72)
for condition in conditions:
    for clf_name in classifiers:
        m = results[condition][clf_name]
        flag = ' ← BEST' if (condition == bc and clf_name == bclf) else ''
        print(f'Model {condition[:1]:<17} {clf_name:<22} '
              f'{m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} {m["f1_weighted"]:>8.4f}{flag}')
    print()

## 20. Conclusion

This notebook completes **Task 2 — Baseline Modelling** for the domain-adaptive sentiment analysis project.

### What Was Done

1. **Loaded and explored** two datasets — GoEmotions (43,404 general-domain samples) and Financial PhraseBank (3,392 domain-specific train samples, 727 test samples)
2. **Analysed** label distributions, text length profiles, and vocabulary overlap between domains
3. **Preprocessed** text through a seven-step normalisation pipeline
4. **Built** TF-IDF + classifier pipelines for three models (Logistic Regression, Linear SVM, ComplementNB)
5. **Ran 9 experiments** across conditions A (general only), B (domain only), and C (mixed)
6. **Evaluated** with accuracy, macro F1, weighted F1, per-class F1, confusion matrices, and normalised confusion matrices
7. **Analysed** TF-IDF vocabulary differences, feature weights, error patterns, and sample efficiency

### Numerical Summary

| Condition | Best Classifier | Accuracy | F1 (Macro) | F1 (Weighted) |
|-----------|----------------|----------|------------|---------------|
| A — General Only | Linear SVM | ~0.561 | ~0.268 | ~0.476 |
| B — Domain Only | Linear SVM | ~0.762 | ~0.563 | ~0.754 |
| C — Mixed | Linear SVM | ~0.752 | ~0.542 | ~0.742 |

### Answer to the Research Question

> **How does training data choice influence model performance?**

Training data choice is the dominant factor at the TF-IDF baseline level. A ~30 pp macro F1 gap exists between Model A and Model B using **identical architectures** — the only difference is training data. This establishes that domain adaptation is necessary and motivates the Transformer-based experiments in Tasks 3–5.

### Next Steps

- **Task 3** — Transformer-based modelling (BERT variants) under the same A/B/C framework; target macro F1 ≥ 0.70
- **Task 4** — Sequential transfer learning (GoEmotions pre-training → FPB fine-tuning) and domain-adaptive pretraining
- **Task 5** — LLM zero-shot and few-shot prompting comparison against these baselines